# Notebook 04 — Genetic Drift and Selection: Extended Models

**Module:** 06 — Genetics and Evolution  
**Notebook:** 04 of 12  
**Prerequisites:** Module 05 NB06 (HWE), NB07 (WF drift), NB08 (selection)  
**Time estimate:** 90 minutes

> The Kimura fixation probability formula derived here is a **Pass 3 reconstruction
> target** — implement from memory, then check against the simulation.

---
## Step 1 — Motivation

Module 05 established that alleles change frequency by drift and selection. This notebook
asks the next question: *given a specific allele starting at frequency p₀ with selection
coefficient s in a population of size N, what is the probability it eventually fixes?*
Kimura's formula answers this analytically. The gap between 'drift removes alleles'
and 'the fixation probability is exactly (1-e^{-2Nsp₀})/(1-e^{-2Ns})' is what this
notebook closes.

---
## Step 3 — Biological Background

**Fixation probability under selection (Kimura 1962):**

For a diploid population of effective size Nₑ, an allele starting at frequency p₀
with selection coefficient s (additive model) has fixation probability:

$$P_{fix} = \frac{1 - e^{-2N_e s p_0}}{1 - e^{-2N_e s}}$$

Special cases:
- s = 0 (neutral): P_fix = p₀ (a neutral allele fixes with probability equal to its
  current frequency — the 'neutral fixation probability')
- p₀ = 1/(2Nₑ) (new mutation): P_fix ≈ 2s for s > 0 (rare beneficial alleles fix
  with probability approximately twice the selection coefficient)
- s very negative: P_fix → 0 (strongly deleterious alleles almost never fix)

**Selective sweep signatures:**
When a beneficial allele sweeps from low frequency to fixation, it drags nearby
sequences along — a **selective sweep**. This creates a region of reduced genetic
diversity flanking the selected site. Detected by:
- iHS (integrated haplotype score)
- XP-EHH (cross-population extended haplotype homozygosity)
- Tajima's D (negative value indicates excess rare alleles, consistent with a sweep)

---
## Step 4 — Mathematical Explanation

**Tajima's D statistic:**
$$D = \frac{\hat{\theta}_{\pi} - \hat{\theta}_{W}}{\sqrt{\text{Var}(\hat{\theta}_{\pi} - \hat{\theta}_{W})}}$$

where:
- θ̂_π = mean pairwise differences between sequences (nucleotide diversity)
- θ̂_W = Watterson estimator = S / a₁ where S = number of segregating sites,
  a₁ = Σᵢ₌₁ⁿ⁻¹ 1/i

Interpretation:
- D < 0: excess of rare alleles → recent selective sweep or population expansion
- D = 0: consistent with neutral evolution
- D > 0: excess of common alleles → balancing selection or population contraction

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Kimura fixation probability
def kimura_fixation_prob(p0: float, s: float, Ne: int) -> float:
    """
    Kimura (1962) fixation probability for a selected allele.

    p0 : initial allele frequency
    s  : selection coefficient (additive model; can be negative)
    Ne : effective population size
    """
    if abs(s) < 1e-10:  # neutral limit: P_fix = p0
        return p0
    exponent = -2 * Ne * s
    numerator   = 1 - np.exp(exponent * p0)
    denominator = 1 - np.exp(exponent)
    return numerator / denominator

# Verify special cases
Ne = 1000
p0_new = 1 / (2 * Ne)  # a single new mutation

print("Kimura fixation probability: special cases")
print(f"  Neutral (s=0, p0=0.01):           P_fix = {kimura_fixation_prob(0.01, 0.0, Ne):.4f} (expected: 0.01)")
print(f"  New neutral (s=0, p0=1/(2Ne)):    P_fix = {kimura_fixation_prob(p0_new, 0.0, Ne):.6f} (expected: {p0_new:.6f})")
print(f"  Beneficial s=0.01 (p0=1/(2Ne)):   P_fix = {kimura_fixation_prob(p0_new, 0.01, Ne):.6f} (expected ~2s={0.02:.4f})")
print(f"  Deleterious s=-0.01 (p0=0.5):     P_fix = {kimura_fixation_prob(0.5, -0.01, Ne):.6f}")

In [ ]:
# Cell 6.2 — Verify Kimura formula against Wright-Fisher simulation
def wright_fisher_fixation_prob(p0, s, Ne, n_trials=5000, n_gen=None, seed=42):
    """Estimate fixation probability by simulation."""
    rng = np.random.default_rng(seed)
    if n_gen is None:
        n_gen = 12 * Ne  # long enough for most alleles to fix or be lost

    fixed = 0
    for _ in range(n_trials):
        p = p0
        for _ in range(n_gen):
            if p <= 0 or p >= 1:
                break
            # Selection
            q = 1 - p
            p_eff = p * (1 + s) / (p * (1 + s) + q) if s != 0 else p
            # Drift
            p = rng.binomial(2 * Ne, p_eff) / (2 * Ne)
        if p >= 1:
            fixed += 1
    return fixed / n_trials

# Compare theory vs. simulation
Ne_sim = 200   # smaller N for faster simulation
p0_sim = 0.05
selection_coefficients = [-0.02, 0.0, 0.005, 0.01, 0.05]

print(f"Fixation probability: theory vs. simulation (Ne={Ne_sim}, p0={p0_sim})")
print(f"  {'s':>7} {'Kimura':>10} {'Simulation':>12} {'Ratio':>8}")
print("  " + "-"*42)
for s in selection_coefficients:
    p_theory = kimura_fixation_prob(p0_sim, s, Ne_sim)
    p_sim    = wright_fisher_fixation_prob(p0_sim, s, Ne_sim, n_trials=2000,
                                            n_gen=20*Ne_sim)
    ratio = p_sim / p_theory if p_theory > 1e-8 else float('nan')
    print(f"  {s:>7.3f} {p_theory:>10.5f} {p_sim:>12.5f} {ratio:>8.3f}")

In [ ]:
# Cell 6.3 — Tajima's D (simplified implementation for understanding)
def tajimas_d(sequences: list) -> dict:
    """
    Compute Tajima's D from aligned DNA sequences.
    sequences: list of strings (aligned, same length)
    """
    n = len(sequences)
    L = len(sequences[0])

    # Count segregating sites
    S = sum(1 for j in range(L) if len(set(s[j] for s in sequences)) > 1)

    # Watterson estimator: theta_W = S / a1
    a1 = sum(1/i for i in range(1, n))
    theta_W = S / a1 if a1 > 0 else 0

    # Nucleotide diversity: mean pairwise differences
    total_diffs = 0
    n_pairs = 0
    for i in range(n):
        for j in range(i+1, n):
            total_diffs += sum(sequences[i][k] != sequences[j][k] for k in range(L))
            n_pairs += 1
    pi = total_diffs / (n_pairs * L) if n_pairs > 0 else 0
    theta_pi = pi * L

    # Simplified variance (full formula in Tajima 1989)
    a2 = sum(1/i**2 for i in range(1, n))
    b1 = (n+1) / (3*(n-1))
    b2 = 2*(n**2+n+3) / (9*n*(n-1))
    c1 = b1 - 1/a1
    c2 = b2 - (n+2)/(a1*n) + a2/a1**2
    e1 = c1 / a1
    e2 = c2 / (a1**2 + a2)
    var_d = e1 * S + e2 * S * (S - 1)

    D = (theta_pi - theta_W) / np.sqrt(var_d) if var_d > 0 else 0
    return {'D': D, 'theta_pi': theta_pi, 'theta_W': theta_W,
            'S': S, 'n': n, 'L': L}

# Example: synthetic sequences under different evolutionary scenarios
rng = np.random.default_rng(7)

def simulate_sequences(n=20, L=500, n_mutations=30, seed=42):
    rng = np.random.default_rng(seed)
    ancestor = ''.join(rng.choice(list('ACGT'), L))
    seqs = [list(ancestor) for _ in range(n)]
    for _ in range(n_mutations):
        pos = rng.integers(L)
        affected = rng.choice(n, size=rng.integers(1, n//2), replace=False)
        alt = rng.choice([b for b in 'ACGT' if b != seqs[0][pos]])
        for i in affected:
            seqs[i][pos] = alt
    return [''.join(s) for s in seqs]

neutral_seqs = simulate_sequences(n=20, L=500, n_mutations=30, seed=42)
sweep_seqs = simulate_sequences(n=20, L=500, n_mutations=5, seed=42)  # fewer variants → sweep-like

r_neutral = tajimas_d(neutral_seqs)
r_sweep   = tajimas_d(sweep_seqs)
print(f"Tajima's D — neutral-like:  D={r_neutral['D']:.3f}  (S={r_neutral['S']})")
print(f"Tajima's D — sweep-like:    D={r_sweep['D']:.3f}  (S={r_sweep['S']}, fewer variants)")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Kimura fixation probability surface + selective sweep illustration
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Fixation probability vs. s for different Ne
ax = axes[0]
s_range = np.linspace(-0.05, 0.1, 300)
for Ne_plot, color in [(50, 'tomato'), (200, 'steelblue'), (1000, 'seagreen')]:
    pfix = [kimura_fixation_prob(0.01, s, Ne_plot) for s in s_range]
    ax.plot(s_range, pfix, color=color, lw=2, label=f'Ne={Ne_plot}')
ax.axhline(0.01, color='gray', ls='--', lw=0.8, label='Neutral limit (P_fix=p0=0.01)')
ax.axvline(0, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Selection coefficient s')
ax.set_ylabel('Fixation probability')
ax.set_title('Kimura fixation probability (p0=0.01):\nLarger Ne = more influence of selection')
ax.legend(fontsize=8)

# Panel 2: Fixation probability as function of starting frequency (neutral vs. selected)
ax = axes[1]
p0_range = np.linspace(0.001, 0.999, 300)
for s_plot, color, label in [(0.0, 'gray', 'Neutral (s=0)'),
                               (0.01, 'steelblue', 's=0.01'),
                               (0.05, 'tomato', 's=0.05'),
                               (-0.01, 'seagreen', 's=-0.01 (deleterious)')]:
    pfix = [kimura_fixation_prob(p, s_plot, 500) for p in p0_range]
    ax.plot(p0_range, pfix, color=color, lw=2, label=label)
ax.plot(p0_range, p0_range, 'k--', lw=1, alpha=0.3, label='P_fix = p0 (reference)')
ax.set_xlabel('Initial allele frequency p0')
ax.set_ylabel('Fixation probability')
ax.set_title('Fixation probability vs. starting frequency (Ne=500)\nBeneficial alleles have advantage at all p0')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `kimura_fixation_prob(p0, s, Ne)` from scratch. Verify the neutral limit:
   as s→0, does it converge to p0? (Use L'Hôpital's rule or just take the limit
   numerically.)
2. A new beneficial mutation arises in a population of Ne=10,000 with s=0.01.
   What is the probability it eventually fixes? How does this compare to a neutral
   new mutation (p0 = 1/2Ne)?
3. Tajima's D is negative in a region of the genome. Name two biological explanations.
   How would you distinguish them using additional data?
4. Why does a selective sweep reduce genetic diversity in a region flanking the selected
   site? What is this region called, and how large is it typically in humans?

---
## Quiz — Active Recall

1. What is Kimura's fixation probability formula? State it from memory.
2. What is the fixation probability of a new neutral mutation?
3. What does Tajima's D measure? What does D < 0 suggest?
4. What is a selective sweep? What genomic signal does it leave?
5. Why does the fixation probability of a beneficial allele approach 2s for new
   mutations (p0 = 1/2Ne)?

---
## Papers Referenced

- Kimura (1968). DOI: 10.1038/217624a0

---
## Reflection

**Date completed:** ____________________

> *[Can you state Kimura's formula from memory and explain each term? If not, close
> the notebook and try to reconstruct it — this is a Pass 3 reconstruction exercise.]*

---
*Next: `05_coalescent_theory_intuition.ipynb`*